# Día 5 · Extracción documental mejorada

Clasifica elementos como riesgo, hecho ocurrido, compromiso, acción correctiva, hallazgo o información contextual. Exige evidencia textual verificable y genera candidatos separados para vigilancia.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-05-extraccion-mejorada
!pip -q install pandas pyarrow openpyxl openai

## 1. Guardar checkpoint en Google Drive
Autoriza el acceso. Si Colab se interrumpe, vuelve a ejecutar el notebook y continuará desde el último chunk completado.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
output_dir=Path('/content/drive/MyDrive/proyecto_riesgos/dia_05_extraccion')
output_dir.mkdir(parents=True,exist_ok=True)
print('Resultados y checkpoint:',output_dir)

## 2. Introducir la clave de OpenAI
La clave permanece únicamente en la sesión de Colab.

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY']=getpass('OPENAI_API_KEY: ')
assert os.environ['OPENAI_API_KEY'].strip(),'La clave está vacía'
print('Clave cargada en la sesión')

## 3. Ejecutar la extracción
Se procesan 345 chunks. La primera ejecución puede tardar entre 20 y 40 minutos. Si se reinicia la celda, se omiten los chunks ya guardados en el checkpoint.

In [ ]:
!python -m src.risk.extract_documentary_items --chunks data/processed/chunks/chunks_recursive.parquet --output-dir "{output_dir}"

## 4. Revisar controles automáticos

In [ ]:
import json,pandas as pd
items=pd.read_csv(output_dir/'documentary_items.csv')
metadata=json.loads((output_dir/'documentary_extraction_metadata.json').read_text())
print(json.dumps(metadata,ensure_ascii=False,indent=2))
display(items['item_type'].value_counts().rename_axis('tipo').reset_index(name='cantidad'))
display(pd.crosstab(items['item_type'],items['surveillance_candidate'],margins=True))
assert metadata['chunks_completed']==345,'La ejecución no está completa'
assert metadata['api_errors']==0,'Existen errores de API; vuelve a ejecutar la celda 3'
assert items['evidence_verified'].eq(1).all(),'Existe evidencia no verificada'
print('Controles completados correctamente')

## 5. Descargar resultados

In [ ]:
import shutil
from google.colab import files
zip_path=shutil.make_archive('/content/resultados_extraccion_mejorada_dia_05','zip',output_dir)
files.download(zip_path)